In [138]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [139]:
def prepare_data_and_scalers(df):
    """
    Computes categorical index mappings and feature scalers 
    for both Model 1 and Model 2 on the full dataset.
    """
    df = df.copy()
    
    # Common Categorical Mappings
    sets = df['set'].astype('category')
    rarities = df['rarity'].astype('category')
    
    set_mapping = dict(enumerate(sets.cat.categories))
    rarity_mapping = dict(enumerate(rarities.cat.categories))
    
    df['set_idx'] = sets.cat.codes.values
    df['rarity_idx'] = rarities.cat.codes.values

    # Feature Engineering
    df['log_price'] = np.log(df['current_price'])
    df['log_start_price'] = (
        df.groupby('card')['log_price']
        .transform('first')
    )
    df['target_log_return'] = df.groupby('card')['log_price'].diff()
    df.dropna(subset=['target_log_return'], inplace=True)
    df['lag_log_return'] = df.groupby('card')['target_log_return'].shift(1)
    df.dropna(subset=['lag_log_return'], inplace=True)
    df['rel_popularity'] = df['popularity_index'] / (df['set_popularity_index'] + 1e-6)
    df['set_momentum'] = df.groupby(['set_idx', 't'])['lag_log_return'].transform('mean')

    # Feature Calculations
    df['log_t'] = np.log1p(df['t'])  # Avoid log(0)
    df['obs_date'] = pd.to_datetime(df['obs_date'])
    df['set_release_date'] = pd.to_datetime(df['set_release_date'])
    df['true_age_months'] = np.maximum(0, (df['obs_date'] - df['set_release_date']).dt.days / 30.4375)
    df['log_true_age'] = np.log1p(df['true_age_months'])
    df['is_mature'] = (df['true_age_months'] >= 18).astype(float)
    df['rel_pop'] = df['popularity_index'] / (df['set_popularity_index'] + 1e-6)

    # Model 1 Scalers (Relative Time Focus)
    scalers_m1 = {
        'log_t_mean': float(df['log_t'].mean()),
        'log_t_std': float(df['log_t'].std()),
        'rel_pop_mean': float(df['rel_pop'].mean()),
        'rel_pop_std': float(df['rel_pop'].std()),
        'lag_mean': float(df['lag_log_return'].mean()),
        'lag_std': float(df['lag_log_return'].std()),
        'mom_mean': float(df['set_momentum'].mean()),
        'mom_std': float(df['set_momentum'].std())
    }

    # Model 2 Scalers (True Age Focus)
    scalers_m2 = {
        'log_age_mean': float(df['log_true_age'].mean()),
        'log_age_std': float(df['log_true_age'].std()),
        'rel_pop_mean': scalers_m1['rel_pop_mean'],
        'rel_pop_std': scalers_m1['rel_pop_std'],
        'lag_mean': scalers_m1['lag_mean'],
        'lag_std': scalers_m1['lag_std'],
        'mom_mean': scalers_m1['mom_mean'],
        'mom_std': scalers_m1['mom_std']
    }

    # Apply scaling for PyMC execution
    df['m1_scaled_log_t'] = (df['log_t'] - scalers_m1['log_t_mean']) / scalers_m1['log_t_std']
    df['m2_scaled_log_age'] = (df['log_true_age'] - scalers_m2['log_age_mean']) / scalers_m2['log_age_std']
    df['scaled_rel_pop'] = (df['rel_pop'] - scalers_m1['rel_pop_mean']) / scalers_m1['rel_pop_std']
    df['scaled_lag'] = (df['lag_log_return'] - scalers_m1['lag_mean']) / scalers_m1['lag_std']
    df['scaled_mom'] = (df['set_momentum'] - scalers_m1['mom_mean']) / scalers_m1['mom_std']

    df['age_x_value'] = df['m2_scaled_log_age'] * df['log_start_price']
    
    df['is_chase_card'] = (df['log_start_price'] > 1.3).astype(np.float32)
    df['age_x_chase'] = df['m2_scaled_log_age'] * df['is_chase_card']

    mappings = {'set': set_mapping, 'rarity': rarity_mapping}
    return df, scalers_m1, scalers_m2, mappings

In [140]:
DATA_DIR = Path("../../data/processed")

df = pd.read_csv(
    DATA_DIR / "model_data.csv"
)

df, scalers_m1, scalers_m2, mappings = prepare_data_and_scalers(df)
n_sets = len(mappings['set'])
n_rarities = len(mappings['rarity'])

In [141]:
def split_train_test(df, n_cards=50, random_state=42):
    card_info = (
        df.groupby(["set", "number"])["true_age_months"]
        .max()
        .reset_index()
    )
    old_cards = card_info[card_info["true_age_months"] > 24]
    mid_cards = card_info[
        (card_info["true_age_months"] >= 12) &
        (card_info["true_age_months"] <= 24)
    ]
    young_cards = card_info[card_info["true_age_months"] < 12]

    test_cards = pd.concat([
        old_cards.sample(n=n_cards, random_state=random_state),
        mid_cards.sample(n=n_cards, random_state=random_state),
        young_cards.sample(n=n_cards, random_state=random_state),
    ])

    test_card_keys = set(
        zip(test_cards["set"], test_cards["number"])
    )

    is_test = df.apply(
        lambda row: (row["set"], row["number"]) in test_card_keys,
        axis=1
    )

    train = df[~is_test].copy()
    test = df[is_test].copy()

    return train, test

In [142]:
def simulate_card_to_t(card_row, router, target_t=15, conviction_threshold=0.05, card_row_at_t=None):
    """
    Takes a card's current market state and iteratively predicts its price 
    path until it reaches target_t (15).
    """
    current_t = card_row['t']
    simulated_price = card_row['current_price']
    
    sim_row = card_row.copy()
    cumulative_log_return = 0.0
    path = [simulated_price]
    
    while current_t < target_t:
        current_t += 1
        sim_row['t'] = current_t
        
        if 'obs_date' in sim_row:
            sim_row['obs_date'] = sim_row['obs_date'] + pd.DateOffset(months=1)
            
        prediction = router.predict(sim_row)
        step_return = prediction['predicted_log_return']
        
        cumulative_log_return += step_return
        simulated_price = simulated_price * np.exp(step_return)
        path.append(simulated_price)
        
        # FIX: Decay the momentum/lag effect toward zero instead of compounding predictions
        sim_row['lag_log_return'] = sim_row['lag_log_return'] * 0.5 
        sim_row['set_momentum'] = sim_row['set_momentum'] * 0.5
        
    def get_signal(cum_return, threshold):
        if cum_return > threshold:
            return "BUY"
        elif cum_return < -threshold:
            return "SELL"
        else:
            return "HOLD"
        
    signal = get_signal(cumulative_log_return, conviction_threshold)

    if card_row_at_t is not None:
        actual_price_at_t = card_row_at_t['current_price']
        actual_signal = get_signal(np.log(actual_price_at_t / card_row['current_price']), conviction_threshold)
    else:
        actual_price_at_t = None
        actual_signal = None
        
    return {
        'set': card_row['set'],
        'number': card_row['number'],
        'card': card_row['card'],
        'start_t': card_row['t'],
        'start_price': card_row['current_price'],
        'projected_t_price': simulated_price,
        'cumulative_return_pct': (np.exp(cumulative_log_return) - 1) * 100,
        'signal': signal,
        'path': path,
        'months_simulated': current_t - card_row['t'],
        'actual_price_at_t_if_available': actual_price_at_t,
        'actual_signal_if_available': actual_signal
    }

def run_cohort_validation(test_df, router, test_at_t, target_ts=[12, 12, 12]):
    test_at_t_lookup = (
        test_at_t
        .set_index(["set", "number"])
    )

    # 1. Segment the Cohorts
    # Vintage: Age >= 24 months
    vintage_pool = test_df[test_df['true_age_months'] >= 24]
    
    # Mature: Age >= 12 and Age < 24
    mature_pool = test_df[(test_df['true_age_months'] >= 12) & (test_df['true_age_months'] < 24)]
    
    # Hype: Age < 12 and currently at a t < 12 (needs forecasting)
    hype_pool = test_df[(test_df['true_age_months'] < 12) & (test_df['t'] < 12)]
    
    results = []
    
    # # 2. Run Simulations
    for _, row in vintage_pool.iterrows():
        key = (row["set"], row["number"])

        if key not in test_at_t_lookup.index:
            continue

        start_row = test_at_t_lookup.loc[key]

        res = simulate_card_to_t(
            row,
            router,
            target_t=target_ts[0],
            card_row_at_t=start_row
        )

        res["cohort"] = "Vintage (24+m)"
        results.append(res)
        
    for _, row in mature_pool.iterrows():
        key = (row["set"], row["number"])

        if key not in test_at_t_lookup.index:
            continue

        start_row = test_at_t_lookup.loc[key]

        res = simulate_card_to_t(
            row,
            router,
            target_t=target_ts[1],
            card_row_at_t=start_row
        )
        res['cohort'] = 'Mature (12-24m)'
        results.append(res)
        
    for _, row in hype_pool.iterrows():
        res = simulate_card_to_t(
            row,
            router,
            target_t=target_ts[2]
        )
        res['cohort'] = 'Hype (New)'
        results.append(res)
        
    results_df = pd.DataFrame(results)
    
    # 3. Print Summary of Signals
    print("--- Conviction Signals by Cohort ---")
    print(results_df.groupby(['cohort', 'signal']).size().unstack(fill_value=0))
    
    return results_df

In [143]:
def evaluate_simulation_accuracy(results_df):
    """
    Computes MAE, RMSE, Log MAE, and Directional Accuracy on simulation results,
    overall and broken down by cohort.
    """
    # 1. Filter out cards that didn't have a matching actual price at target t
    eval_df = results_df.dropna(subset=['actual_price_at_t_if_available']).copy()
    
    if eval_df.empty:
        print("No actual prices available for validation.")
        return None

    y_start = eval_df['start_price'].values
    y_pred = eval_df['projected_t_price'].values
    y_true = eval_df['actual_price_at_t_if_available'].values
    
    # 2. Dollar Error Metrics
    mae_dollar = np.mean(np.abs(y_pred - y_true))
    rmse_dollar = np.sqrt(np.mean((y_pred - y_true) ** 2))
    
    # 3. Log Return Error (Percentage Error Scale)
    pred_log_ret = np.log(y_pred / y_start)
    true_log_ret = np.log(y_true / y_start)
    mae_log = np.mean(np.abs(pred_log_ret - true_log_ret))
    
    # 4. Directional Accuracy
    # Sign of predicted move (up=1, flat=0, down=-1) vs actual move
    pred_dir = np.sign(y_pred - y_start)
    true_dir = np.sign(y_true - y_start)
    dir_accuracy = np.mean(pred_dir == true_dir) * 100
    
    # 5. Output Overall Metrics
    print(f"=== Overall Evaluation (N = {len(eval_df)}) ===")
    print(f"Dollar MAE:            ${mae_dollar:.4f}")
    print(f"Dollar RMSE:           ${rmse_dollar:.4f}")
    print(f"Log Return MAE:        {mae_log:.4f}")
    print(f"Directional Accuracy:  {dir_accuracy:.2f}%\n")
    
    # 6. Break down by Cohort
    if 'cohort' in eval_df.columns:
        cohort_records = []
        for cohort_name, group in eval_df.groupby('cohort'):
            g_start = group['start_price'].values
            g_pred = group['projected_t_price'].values
            g_true = group['actual_price_at_t_if_available'].values
            
            c_mae = np.mean(np.abs(g_pred - g_true))
            c_rmse = np.sqrt(np.mean((g_pred - g_true) ** 2))
            c_dir = np.mean(np.sign(g_pred - g_start) == np.sign(g_true - g_start)) * 100
            
            cohort_records.append({
                'Cohort': cohort_name,
                'Sample Size': len(group),
                'Dollar MAE': f"${c_mae:.2f}",
                'Dollar RMSE': f"${c_rmse:.2f}",
                'Directional Acc.': f"{c_dir:.1f}%"
            })
            
        summary_table = pd.DataFrame(cohort_records)
        print("--- Cohort Breakdown ---")
        print(summary_table.to_string(index=False))
        
    return {
        'mae_dollar': mae_dollar,
        'rmse_dollar': rmse_dollar,
        'mae_log': mae_log,
        'directional_accuracy': dir_accuracy
    }

In [153]:
from IPython.display import HTML
import json 

card_image = json.load(open(DATA_DIR / "card_image_urls.json", "r"))
def display_simulation_table(results_df, card_image_dict):
    df = results_df.copy()
    
    df["image_url"] =  df["set"] + "/" + df["number"].astype(str)
    df["image_url"] = df["image_url"].map(card_image_dict)
    df['Image'] = df['image_url'].apply(
        lambda url: f'<img src="{url}" style="height: 75px; max-width: 75px; object-fit: contain; border-radius: 4px;" />' 
        if pd.notna(url) and str(url).startswith('http') 
        else '<span style="color: #9ca3af;">No Image</span>'
    )
    
    # 2. Format Currency and Return Percentages
    df['Months Simulated'] = df['months_simulated'].apply(lambda x: f"{x} month{'s' if x != 1 else ''}")
    df['Current Price'] = df['start_price'].apply(lambda x: f"${x:,.2f}")
    df['Projected'] = df['projected_t_price'].apply(lambda x: f"${x:,.2f}") 
    df['Actual Price at t'] = df['actual_price_at_t_if_available'].apply(lambda x: f"${x:,.2f}" if pd.notna(x) else '<span style="color: #9ca3af;">N/A</span>')
    df['Price Difference'] = df.apply(
        lambda row: f"${row['projected_t_price'] - row['actual_price_at_t_if_available']:+,.2f}" if pd.notna(row['actual_price_at_t_if_available']) else '<span style="color: #9ca3af;">N/A</span>',
        axis=1
    )
    df['Return'] = df['cumulative_return_pct'].apply(lambda x: f"{x:+.2f}%")
    
    # 3. Add Color Badges for Signals
    def format_badge(signal):
        badge_styles = {
            'BUY': 'background-color: #16a34a; color: white;',
            'SELL': 'background-color: #dc2626; color: white;',
            'HOLD': 'background-color: #4b5563; color: white;'
        }
        style = badge_styles.get(signal, 'background-color: #9ca3af; color: white;')
        return f'<span style="{style} padding: 4px 10px; border-radius: 12px; font-weight: 600; font-size: 11px; letter-spacing: 0.5px;">{signal}</span>'
    
    df['Signal'] = df['signal'].apply(format_badge)
    df['Actual Signal'] = df['actual_signal_if_available'].apply(lambda x: format_badge(x) if pd.notna(x) else '<span style="color: #9ca3af;">N/A</span>')
    
    # 4. Select and Rename Display Columns
    display_cols = {
        'Image': 'Card Image',
        'card': 'Card Key (Series/Number)',
        'cohort': 'Cohort',
        'Months Simulated': 'Months Simulated',
        'Current Price': 'Current Price',
        'Projected': 'Projected Price',
        'Price Difference': 'Price Difference',
        'Actual Price at t': 'Actual Price at t',
        'Return': 'Exp. Return (%)',
        'Signal': 'Signal',
        'Actual Signal': 'Actual Signal'
    }
    
    table_df = df[list(display_cols.keys())].rename(columns=display_cols)
    
    # 5. Apply Custom CSS Styling for Notebook Output
    html_content = table_df.to_html(escape=False, index=False)
    
    styled_table = f"""
    <style>
        .card-sim-table {{
            border-collapse: collapse;
            width: 100%;
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
            font-size: 13px;
        }}
        .card-sim-table th {{
            background-color: #1f2937;
            color: #f9fafb;
            text-align: left;
            padding: 10px 14px;
            font-weight: 600;
        }}
        .card-sim-table td {{
            padding: 8px 14px;
            border-bottom: 1px solid #e5e7eb;
            vertical-align: middle;
        }}
        .card-sim-table tr:hover {{
            background-color: #f3f4f6;
        }}
    </style>
    {html_content.replace('class="dataframe"', 'class="card-sim-table"')}
    """
    
    return HTML(styled_table)

In [145]:
class WeightedDirectionalMSELoss(nn.Module):
    def __init__(self, penalty_weight: float = 3.0, epsilon: float = 0.01):
        super().__init__()
        self.penalty_weight = penalty_weight
        self.epsilon = epsilon

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor, sample_weights: torch.Tensor) -> torch.Tensor:
        # Squared error weighted by card value/tier
        squared_errors = (y_pred - y_true) ** 2
        weighted_mse = torch.mean(sample_weights * squared_errors)
        
        # Mask out noise (only evaluate direction for moves exceeding epsilon)
        significant_mask = (torch.abs(y_true) > self.epsilon).float()
        
        # Penalize opposite signs: y_pred * y_true < 0
        sign_product = y_pred * y_true
        direction_penalty = torch.relu(-sign_product) * significant_mask
        weighted_direction_penalty = torch.mean(sample_weights * direction_penalty)
        
        return weighted_mse + (self.penalty_weight * weighted_direction_penalty)

In [146]:
class VintageAwareReturnNet(nn.Module):
    def __init__(self, num_sets, num_rarities, num_continuous_features, emb_dim=8, max_log_ret=1.5):
        super().__init__()
        self.max_log_ret = max_log_ret  # 1.5 log-return ≈ -78% to +348% max monthly move
        self.set_embed = nn.Embedding(num_sets, emb_dim)
        self.rarity_embed = nn.Embedding(num_rarities, emb_dim)
        
        in_dim = (emb_dim * 2) + num_continuous_features
        
        self.fc1 = nn.Linear(in_dim, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(64, 32)
        self.out_head = nn.Linear(32, 1)
        
        nn.init.zeros_(self.out_head.bias)
        nn.init.xavier_uniform_(self.out_head.weight)
        
    def forward(self, x_cat, x_cont):
        set_emb = self.set_embed(x_cat[:, 0])
        rarity_emb = self.rarity_embed(x_cat[:, 1])
        x = torch.cat([set_emb, rarity_emb, x_cont], dim=1)
        
        h = self.relu(self.bn1(self.fc1(x)))
        h = self.dropout(h)
        h = self.relu(self.fc2(h))
        
        raw_out = self.out_head(h).squeeze(-1)
        # Bounded between [-max_log_ret, +max_log_ret]
        return torch.tanh(raw_out) * self.max_log_ret

In [147]:
class TabularDataset(Dataset):
    def __init__(self, x_cat, x_cont, y, weights):
        self.x_cat = torch.tensor(x_cat, dtype=torch.long)
        self.x_cont = torch.tensor(x_cont, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.weights = torch.tensor(weights, dtype=torch.float32)
        
    def __len__(self):
        return len(self.y)
        
    def __getitem__(self, idx):
        return self.x_cat[idx], self.x_cont[idx], self.y[idx], self.weights[idx]
    
def train_pytorch_model(df, continuous_cols, epochs=40, lr=0.001):
    df = df.copy()
    
    # Ensure interaction features & weights exist
    if 'log_start_price' not in df.columns:
        df['log_start_price'] = np.log1p(df['start_price'])
    
    # 1. Feature Mappings
    set_codes, set_uniques = pd.factorize(df['set'])
    rarity_codes, rarity_uniques = pd.factorize(df['rarity'])
    
    mappings = {
        'set': {name: idx for idx, name in enumerate(set_uniques)},
        'rarity': {name: idx for idx, name in enumerate(rarity_uniques)}
    }
    
    # 2. Continuous Feature Scaling
    scalers = {}
    x_cont_scaled = np.zeros((len(df), len(continuous_cols)), dtype=np.float32)
    for i, col in enumerate(continuous_cols):
        mean = df[col].mean()
        std = df[col].std() + 1e-6
        scalers[col] = {'mean': mean, 'std': std}
        x_cont_scaled[:, i] = (df[col] - mean) / std
        
    x_cat = np.column_stack([set_codes, rarity_codes])
    y = df['target_log_return'].values
    
    # Fix 1: Convert pandas Series to numpy array via .values
    sample_weights = (1.0 + df['log_start_price']).values
    
    # Fix 2: Pass sample_weights to TabularDataset
    dataset = TabularDataset(x_cat, x_cont_scaled, y, sample_weights)
    loader = DataLoader(dataset, batch_size=64, shuffle=True)
    
    model = VintageAwareReturnNet(
        num_sets=len(set_uniques), 
        num_rarities=len(rarity_uniques), 
        num_continuous_features=len(continuous_cols)
    )
    
    # Instantiate Loss without fixed dataset weights
    criterion = WeightedDirectionalMSELoss(penalty_weight=2.0, epsilon=0.01)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    model.train()
    for epoch in range(epochs):
        for batch_cat, batch_cont, batch_y, batch_w in loader:
            optimizer.zero_grad()
            preds = model(batch_cat, batch_cont)
            loss = criterion(preds, batch_y, batch_w)
            loss.backward()
            
            # Clip exploding gradients before optimizer step
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
    return model, scalers, mappings

In [148]:
class PyTorchModelRouter:
    def __init__(self, model, scalers, mappings, continuous_cols):
        self.model = model.eval()
        self.scalers = scalers
        self.mappings = mappings
        self.continuous_cols = continuous_cols

    def predict(self, row):
        # 1. Encode Categoricals with Fallback for unseen values
        set_idx = self.mappings['set'].get(row['set'], 0)
        rarity_idx = self.mappings['rarity'].get(row['rarity'], 0)
        x_cat = torch.tensor([[set_idx, rarity_idx]], dtype=torch.long)
        
        # 2. Scale Continuous Features
        cont_vals = []
        for col in self.continuous_cols:
            val = row[col]
            mean = self.scalers[col]['mean']
            std = self.scalers[col]['std']
            cont_vals.append((val - mean) / std)
            
        x_cont = torch.tensor([cont_vals], dtype=torch.float32)
        
        # 3. Inference
        with torch.no_grad():
            predicted_return = self.model(x_cat, x_cont).item()
            
        return {
            'card': row['card'],
            't': row['t'],
            'predicted_log_return': predicted_return,
            'model_used': 'PyTorch Deep Learning (Directional Loss)'
        }

In [150]:
ts = [6, 7, 8, 9, 10, 11, 12]
continuous_features = ['t', 'log_true_age', 'is_mature', 'lag_log_return', 'set_momentum']

finished_list = []
unfinished_list = []

for t in ts:
    print(f"--- Simulation Results for t={t} ---")
    train_df, test_df = split_train_test(df, n_cards=100, random_state=None)

    test_selected = (
        test_df[test_df["t"] <= t]
        .sort_values("t")
        .groupby(["set", "number"], as_index=False)
        .last()
    )

    target_t = 12 if t < 12 else 15

    test_at_t = (
        test_df[test_df["t"] == target_t]
        .sort_values("t")
        .groupby(["set", "number"], as_index=False)
        .last()
    )

    pt_model, scalers, mappings = train_pytorch_model(
        train_df, 
        continuous_cols=continuous_features, 
        epochs=40
    )

    dl_router = PyTorchModelRouter(pt_model, scalers, mappings, continuous_features)
    simulation_results = run_cohort_validation(
        test_df=test_selected,
        router=dl_router,
        test_at_t=test_at_t
    )

    print(evaluate_simulation_accuracy(simulation_results))

    finished_simulation_results = simulation_results[simulation_results['actual_price_at_t_if_available'].notnull()].copy()
    finished_simulation_results = finished_simulation_results[finished_simulation_results["start_price"] > 10]
    finished_list.append(finished_simulation_results)

    unfinished_simulation_results = simulation_results[simulation_results['actual_price_at_t_if_available'].isnull()].copy()
    unfinished_simulation_results = unfinished_simulation_results[unfinished_simulation_results["start_price"] > 10]
    unfinished_list.append(unfinished_simulation_results)

--- Simulation Results for t=6 ---
--- Conviction Signals by Cohort ---
signal           BUY  HOLD  SELL
cohort                          
Hype (New)        20    16    64
Mature (12-24m)   42    55     3
Vintage (24+m)    78    18     4
=== Overall Evaluation (N = 200) ===
Dollar MAE:            $13.3502
Dollar RMSE:           $67.6607
Log Return MAE:        0.4897
Directional Accuracy:  72.00%

--- Cohort Breakdown ---
         Cohort  Sample Size Dollar MAE Dollar RMSE Directional Acc.
Mature (12-24m)          100      $9.66      $26.01            56.0%
 Vintage (24+m)          100     $17.04      $92.08            88.0%
{'mae_dollar': np.float64(13.350183855017121), 'rmse_dollar': np.float64(67.66067464512098), 'mae_log': np.float64(0.4897137945512152), 'directional_accuracy': np.float64(72.0)}
--- Simulation Results for t=7 ---
--- Conviction Signals by Cohort ---
signal           BUY  HOLD  SELL
cohort                          
Hype (New)         0    41    59
Mature (12-24m)   47

In [ ]:
display_simulation_table(finished_list[0], card_image)

Card Image,Card Key (Series/Number),Cohort,Months Simulated,Current Price,Projected Price,Price Difference,Actual Price at t,Exp. Return (%),Signal,Actual Signal
,Charizard ex,Vintage (24+m),6 months,$36.19,$39.65,$-1.51,$41.16,+9.57%,BUY,BUY
,Blastoise ex,Vintage (24+m),6 months,$105.56,$117.51,$-22.24,$139.75,+11.32%,BUY,BUY
,Erika's Invitation,Vintage (24+m),6 months,$15.71,$17.51,$-0.34,$17.85,+11.48%,BUY,BUY
,Single Strike Urshifu VMAX,Vintage (24+m),6 months,$32.28,$35.13,$+3.92,$31.21,+8.83%,BUY,HOLD
,Ice Rider Calyrex V,Vintage (24+m),6 months,$16.59,$19.46,$-4.38,$23.84,+17.30%,BUY,BUY
,Espeon V,Vintage (24+m),6 months,$181.41,$209.93,$-30.37,$240.30,+15.72%,BUY,BUY
,Sylveon V,Vintage (24+m),6 months,$31.53,$36.52,$+0.29,$36.23,+15.83%,BUY,BUY
,Sylveon V,Vintage (24+m),6 months,$138.60,$161.15,$-45.35,$206.50,+16.27%,BUY,BUY
,Leafeon VMAX,Vintage (24+m),6 months,$30.48,$33.00,$-10.20,$43.20,+8.25%,BUY,BUY
,Gyarados VMAX,Vintage (24+m),6 months,$35.95,$39.03,$-10.59,$49.62,+8.57%,BUY,BUY
